In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

def show(name, x):
    print(f"{name}: shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

# Practical 3: Rotary Positional Embeddings (RoPE)

In the previous practical you implemented multi-head attention. In this notebook we add **rotary positional embeddings (RoPE)** to attention and test that the implementation behaves as expected.

We proceed in small steps:
1. Warm up with ordinary 2D rotations.
2. Implement the core RoPE helpers.
3. Test the key algebraic properties.
4. Insert RoPE into multi-head self-attention.
5. Run end-to-end checks.


## 1) Warm-up: 2D rotation

A 2D rotation by angle $\theta$ is

$$
R_\theta = \begin{pmatrix}
\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta
\end{pmatrix}.
$$

RoPE applies exactly this idea, but independently to consecutive pairs of coordinates.


In [2]:
def rotation_matrix(theta: float, device=device):
    c = math.cos(theta)
    s = math.sin(theta)
    return torch.tensor([[c, -s], [s, c]], dtype=torch.float32, device=device)


theta = 0.7
R = rotation_matrix(theta)
x = torch.tensor([1.5, -0.3], device=device)
y = R @ x

print('R =')
print(R)
print('||x|| =', x.norm().item())
print('||R x|| =', y.norm().item())
assert torch.allclose(x.norm(), y.norm(), atol=1e-6)


R =
tensor([[ 0.7648, -0.6442],
        [ 0.6442,  0.7648]])
||x|| = 1.5297058820724487
||R x|| = 1.5297058820724487


## 2) RoPE on pairs of coordinates

Suppose the head dimension is `d_head`, and assume it is even. We split each vector into consecutive pairs

$$
(x_0, x_1), (x_2, x_3), \ldots, (x_{d-2}, x_{d-1}),
$$

and rotate the $i$-th pair by an angle that depends on:
- the token position `p`,
- the frequency index `i`.

A standard choice is

$$
\theta_i = 10000^{-2i/d_{\text{head}}},
\qquad
\text{angle}(p, i) = p \theta_i.
$$


In [3]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Input: x of shape (..., d) with even d
    Output: (-x_1, x_0, -x_3, x_2, ..., -x_{d-1}, x_{d-2})
    so that pairwise rotation can be written as
        x_rot = x * cos + rotate_half(x) * sin
    where cos/sin are broadcastable over the last dimension.
    """
    d = x.shape[-1]
    assert d % 2 == 0, 'RoPE requires an even last dimension'
    x_pairs = x.view(*x.shape[:-1], d // 2, 2)
    x1 = x_pairs[..., 0]
    x2 = x_pairs[..., 1]
    y = torch.stack((-x2, x1), dim=-1)
    return y.view_as(x)

x = torch.tensor([1., 2., 3., 4.], device=device)
y = rotate_half(x)
print('x =', x)
print('rotate_half(x) =', y)
assert torch.equal(y, torch.tensor([-2., 1., -4., 3.], device=device))


x = tensor([1., 2., 3., 4.])
rotate_half(x) = tensor([-2.,  1., -4.,  3.])


In [4]:
def build_rope_cache(seq_len: int, d_head: int, base: float = 10000.0, device=device):
    """
    Returns cos, sin each of shape (seq_len, d_head).
    The same angle is repeated across each consecutive coordinate pair.
    """
    assert d_head % 2 == 0, 'RoPE requires even d_head'

    half = d_head // 2
    freq_idx = torch.arange(half, dtype=torch.float32, device=device)
    inv_freq = base ** (-2 * freq_idx / d_head)

    positions = torch.arange(seq_len, dtype=torch.float32, device=device)
    angles = torch.outer(positions, inv_freq)

    cos = torch.cos(angles).repeat_interleave(2, dim=-1)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)
    return cos, sin


seq_len = 6
d_head = 8
cos, sin = build_rope_cache(seq_len, d_head)
show('cos', cos)
show('sin', sin)
assert cos.shape == (seq_len, d_head)
assert sin.shape == (seq_len, d_head)
assert torch.allclose(cos[0], torch.ones(d_head, device=device))
assert torch.allclose(sin[0], torch.zeros(d_head, device=device))


cos: shape=(6, 8), dtype=torch.float32, device=cpu
sin: shape=(6, 8), dtype=torch.float32, device=cpu


In [5]:
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x:   (B, H, T, d_head)
    cos: (T, d_head)
    sin: (T, d_head)
    """
    assert x.ndim == 4
    B, H, T, d_head = x.shape
    assert cos.shape == (T, d_head)
    assert sin.shape == (T, d_head)

    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return x * cos + rotate_half(x) * sin


B, H, T, d = 2, 3, 5, 8
x = torch.randn(B, H, T, d, device=device)
cos, sin = build_rope_cache(T, d)
y = apply_rope(x, cos, sin)
show('x', x)
show('y', y)
assert y.shape == x.shape


x: shape=(2, 3, 5, 8), dtype=torch.float32, device=cpu
y: shape=(2, 3, 5, 8), dtype=torch.float32, device=cpu


## 3) Basic tests

A good implementation should satisfy some immediate sanity checks:
- shape is preserved,
- at position `p = 0`, nothing changes,
- pairwise Euclidean norms are preserved, because rotations are orthogonal.


In [6]:
B, H, T, d = 2, 4, 7, 8
x = torch.randn(B, H, T, d, device=device)
cos, sin = build_rope_cache(T, d)
y = apply_rope(x, cos, sin)

assert y.shape == x.shape
assert torch.allclose(y[:, :, 0], x[:, :, 0], atol=1e-6)

x_pairs = x.view(B, H, T, d // 2, 2)
y_pairs = y.view(B, H, T, d // 2, 2)
x_pair_norms = x_pairs.norm(dim=-1)
y_pair_norms = y_pairs.norm(dim=-1)
assert torch.allclose(x_pair_norms, y_pair_norms, atol=1e-5)

print('Basic RoPE sanity checks passed.')


Basic RoPE sanity checks passed.


## 4) Compare with an explicit block-diagonal rotation

For one token position `p`, RoPE acts like a block-diagonal matrix with a 2x2 rotation block on each consecutive pair.


In [ ]:
def explicit_rope_matrix(position: int, d_head: int, base: float = 10000.0, device=device):
    assert d_head % 2 == 0
    half = d_head // 2
    freq_idx = torch.arange(half, dtype=torch.float32, device=device)
    inv_freq = base ** (-2 * freq_idx / d_head)
    angles = position * inv_freq

    M = torch.zeros(d_head, d_head, dtype=torch.float32, device=device)
    for i, angle in enumerate(angles):
        c = torch.cos(angle)
        s = torch.sin(angle)
        j = 2 * i
        M[j, j] = c
        M[j, j + 1] = -s
        M[j + 1, j] = s
        M[j + 1, j + 1] = c
    return M


d = 8
p = 3
x = torch.randn(d, device=device)
M = explicit_rope_matrix(p, d)
y1 = M @ x

cos, sin = build_rope_cache(seq_len=p + 1, d_head=d)
x_batched = x.view(1, 1, 1, d)
y2 = apply_rope(x_batched, cos[p:p+1], sin[p:p+1]).view(d)

assert torch.allclose(y1, y2, atol=1e-5)
print('Explicit matrix check passed.')

Explicit matrix check passed.


## 5) The key relative-position identity

The reason RoPE is useful is that dot products between rotated queries and keys depend naturally on **relative position differences**.

In one 2D block, if we rotate `q` by angle `a` and `k` by angle `b`, then

$$
\langle R_a q, R_b k \rangle = \langle q, R_{b-a} k \rangle.
$$

So the interaction depends on `b-a`, not on `a` and `b` separately.


In [8]:
d = 8
p = 2
q_pos = 5

q = torch.randn(d, device=device)
k = torch.randn(d, device=device)

cos, sin = build_rope_cache(seq_len=max(p, q_pos) + 1, d_head=d)

q_rot = apply_rope(q.view(1, 1, 1, d), cos[q_pos:q_pos+1], sin[q_pos:q_pos+1]).view(d)
k_rot = apply_rope(k.view(1, 1, 1, d), cos[p:p+1], sin[p:p+1]).view(d)
lhs = torch.dot(q_rot, k_rot)

M_rel = explicit_rope_matrix(position=p - q_pos, d_head=d)
rhs = torch.dot(q, M_rel @ k)

print('lhs =', lhs.item())
print('rhs =', rhs.item())
assert torch.allclose(lhs, rhs, atol=1e-5)


lhs = -0.012514877133071423
rhs = -0.012514746747910976


## 6) Add RoPE to multi-head self-attention

We now implement a minimal multi-head self-attention layer with RoPE applied to queries and keys before the attention scores are formed.


In [7]:
class MultiHeadAttentionWithRoPE(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, bias: bool = True):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim % 2 == 0, 'RoPE requires even head dimension'

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=bias)

    def forward(self, x: torch.Tensor, is_causal: bool = True):
        B, T, D = x.shape
        H = self.num_heads
        d = self.head_dim

        q = self.W_q(x).view(B, T, H, d).transpose(1, 2)
        k = self.W_k(x).view(B, T, H, d).transpose(1, 2)
        v = self.W_v(x).view(B, T, H, d).transpose(1, 2)

        cos, sin = build_rope_cache(T, d, device=x.device)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)

        if is_causal:
            mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.W_o(out)
        return out, attn


B, T, D, H = 2, 6, 32, 4
x = torch.randn(B, T, D, device=device)
mha = MultiHeadAttentionWithRoPE(embed_dim=D, num_heads=H).to(device)
out, attn = mha(x)
show('out', out)
show('attn', attn)
assert out.shape == (B, T, D)
assert attn.shape == (B, H, T, T)


out: shape=(2, 6, 32), dtype=torch.float32, device=cpu
attn: shape=(2, 4, 6, 6), dtype=torch.float32, device=cpu


## 7) MHA + RoPE tests

When `is_causal=True`, attention from position `t` should never go to positions larger than `t`.


In [8]:
B, T, D, H = 1, 5, 16, 4
x = torch.randn(B, T, D, device=device)
mha = MultiHeadAttentionWithRoPE(embed_dim=D, num_heads=H).to(device)
_, attn = mha(x, is_causal=True)

upper_triangle = torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)
assert torch.allclose(attn[..., upper_triangle], torch.zeros_like(attn[..., upper_triangle]), atol=1e-6)
print('Causal masking check passed.')

Causal masking check passed.


Verify that gradients flow through the RoPE attention layer.


In [9]:
B, T, D, H = 2, 7, 32, 4
x = torch.randn(B, T, D, device=device, requires_grad=True)
mha = MultiHeadAttentionWithRoPE(embed_dim=D, num_heads=H).to(device)

out, attn = mha(x, is_causal=True)
loss = out.pow(2).mean()
loss.backward()

assert x.grad is not None
assert torch.isfinite(x.grad).all()

for name, param in mha.named_parameters():
    assert param.grad is not None, f'No grad for {name}'
    assert torch.isfinite(param.grad).all(), f'Non-finite grad for {name}'
print('Backward pass check passed.')

Backward pass check passed.
